In [1]:
## SETUP & LOAD
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import sys, os

sys.path.insert(0, os.path.abspath('..'))
import config

engine = create_engine(
    f"mysql+pymysql://{config.DB_CONFIG['user']}:{config.DB_CONFIG['password']}"
    f"@{config.DB_CONFIG['host']}:{config.DB_CONFIG['port']}/{config.DB_CONFIG['database']}"
)

# Only load columns needed — avoids memory crash
dp = pd.read_sql(
    "SELECT ticker, trade_date, high_price, low_price, close_price FROM daily_prices ORDER BY ticker, trade_date",
    engine, parse_dates=['trade_date']
)
# Downcast floats — halves memory usage
for col in ['high_price', 'low_price', 'close_price']:
    dp[col] = pd.to_numeric(dp[col], downcast='float')

sf = pd.read_sql(
    "SELECT ticker, daily_return FROM stock_features WHERE daily_return IS NOT NULL",
    engine
)
sf['daily_return'] = pd.to_numeric(sf['daily_return'], downcast='float')

st = pd.read_sql("SELECT ticker, company_name, market, sector FROM stocks", engine)

print(f"daily_prices  : {len(dp):,} rows | {dp.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"stock_features: {len(sf):,} rows")

daily_prices  : 289,656 rows | 10.3 MB
stock_features: 289,577 rows


In [2]:
## ANNUALIZED VOLATILITY
# std of daily returns × √252 × 100
# √252 converts daily std to annual (252 trading days per year)
vol = (sf.groupby('ticker')['daily_return']
         .std()
         .mul(np.sqrt(252) * 100)
         .round(2)
         .reset_index()
         .rename(columns={'daily_return': 'ann_volatility_pct'}))

print(f"Computed for {len(vol)} stocks")
print("\nTop 5 most volatile:")
print(vol.nlargest(5, 'ann_volatility_pct').to_string(index=False))

Computed for 79 stocks

Top 5 most volatile:
     ticker  ann_volatility_pct
       TSLA           57.380001
        AMD           55.770000
ADANIENT.NS           51.939999
       NFLX           49.990002
       NVDA           45.439999


In [3]:
## AVERAGE TRUE RANGE (ATR-14)
dp_s = dp.sort_values(['ticker', 'trade_date']).copy()
dp_s['prev_close'] = dp_s.groupby('ticker')['close_price'].shift(1)

# Vectorized True Range — max of three measures
hl  = dp_s['high_price'] - dp_s['low_price']
hpc = (dp_s['high_price'] - dp_s['prev_close']).abs()
lpc = (dp_s['low_price']  - dp_s['prev_close']).abs()
dp_s['tr'] = pd.concat([hl, hpc, lpc], axis=1).max(axis=1)

# 14-day rolling ATR, normalised as % of close price
dp_s['atr_14'] = dp_s.groupby('ticker')['tr'].transform(
    lambda x: x.rolling(14, min_periods=14).mean()
)
dp_s['atr_pct'] = (dp_s['atr_14'] / dp_s['close_price'] * 100)

atr = (dp_s.groupby('ticker')['atr_pct']
           .mean().round(2).reset_index()
           .rename(columns={'atr_pct': 'avg_atr_pct'}))

print(f"ATR computed for {len(atr)} stocks")
print("\nTop 5 highest ATR:")
print(atr.nlargest(5, 'avg_atr_pct').to_string(index=False))

ATR computed for 79 stocks

Top 5 highest ATR:
       ticker  avg_atr_pct
  ADANIENT.NS         4.57
         TSLA         4.54
          AMD         4.48
SHRIRAMFIN.NS         3.84
         NFLX         3.67


In [4]:
## MAX DRAWDOWN
# Worst peak-to-trough decline ever experienced by the stock
def max_drawdown(prices):
    roll_max = prices.cummax()
    return ((prices - roll_max) / roll_max * 100).min().round(2)

mdd = (dp.sort_values(['ticker', 'trade_date'])
         .groupby('ticker')['close_price']
         .apply(max_drawdown)
         .reset_index()
         .rename(columns={'close_price': 'max_drawdown_pct'}))

print(f"Max drawdown computed for {len(mdd)} stocks")
print("\nTop 5 worst drawdowns:")
print(mdd.nsmallest(5, 'max_drawdown_pct').to_string(index=False))

Max drawdown computed for 79 stocks

Top 5 worst drawdowns:
       ticker  max_drawdown_pct
INDUSINDBK.NS        -85.029999
          AMD        -84.059998
         NFLX        -81.989998
  ADANIENT.NS        -81.029999
      ONGC.NS        -76.790001


In [5]:
## COMBINE + RANK + SAVE
vol_df = (vol.merge(atr, on='ticker')
             .merge(mdd, on='ticker')
             .merge(st,  on='ticker'))

# Rank within each market (1 = most volatile)
vol_df['volatility_rank'] = (
    vol_df.groupby('market')['ann_volatility_pct']
          .rank(ascending=False, method='dense').astype(int)
)

# Categorize — vectorized, no groupby.apply, market column stays intact
market_totals = vol_df.groupby('market')['ticker'].transform('count')
pct_rank      = vol_df['volatility_rank'] / market_totals
vol_df['volatility_category'] = np.where(pct_rank <= 0.33, 'High',
                                 np.where(pct_rank <= 0.66, 'Medium', 'Low'))

# Create table and save
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS stock_volatility (
            vol_id              INT AUTO_INCREMENT PRIMARY KEY,
            ticker              VARCHAR(20) NOT NULL UNIQUE,
            company_name        VARCHAR(100),
            market              ENUM('India','US'),
            sector              VARCHAR(50),
            ann_volatility_pct  DECIMAL(8,2),
            avg_atr_pct         DECIMAL(8,2),
            max_drawdown_pct    DECIMAL(8,2),
            volatility_rank     INT,
            volatility_category VARCHAR(10)
        )
    """))
    conn.execute(text("TRUNCATE TABLE stock_volatility"))
    conn.commit()

cols = ['ticker','company_name','market','sector',
        'ann_volatility_pct','avg_atr_pct',
        'max_drawdown_pct','volatility_rank','volatility_category']
vol_df[cols].to_sql('stock_volatility', engine, if_exists='append', index=False)
print(f"Saved {len(vol_df)} rows to stock_volatility")

Saved 79 rows to stock_volatility


In [6]:
## TOP VOLATILE STOCKS
for market in ['India', 'US']:
    top = vol_df[vol_df['market'] == market].nsmallest(10, 'volatility_rank')
    print(f"\nTOP 10 MOST VOLATILE — {market}")
    print(top[['ticker','company_name','sector',
               'ann_volatility_pct','avg_atr_pct',
               'max_drawdown_pct','volatility_category']].to_string(index=False))

print("\nTOP 10 ACROSS BOTH MARKETS")
print(vol_df.nlargest(10, 'ann_volatility_pct')
           [['ticker','company_name','market','sector','ann_volatility_pct','volatility_category']]
           .to_string(index=False))


TOP 10 MOST VOLATILE — India
       ticker        company_name      sector  ann_volatility_pct  avg_atr_pct  max_drawdown_pct volatility_category
  ADANIENT.NS   Adani Enterprises Infra/Other           51.939999         4.57        -81.029999                High
SHRIRAMFIN.NS     Shriram Finance     Finance           41.320000         3.84        -71.870003                High
  HINDALCO.NS Hindalco Industries       Metal           39.820000         3.63        -74.320000                High
INDUSINDBK.NS       IndusInd Bank     Banking           38.900002         3.30        -85.029999                High
ADANIPORTS.NS   Adani Ports & SEZ Infra/Other           38.700001         3.58        -53.240002                High
  JSWSTEEL.NS           JSW Steel       Metal           36.860001         3.43        -65.820000                High
 TATASTEEL.NS          Tata Steel       Metal           36.240002         3.23        -69.510002                High
BAJFINANCE.NS       Bajaj Finance 